# Policy for Amazon Bedrock AgentCore - 엔드 투 엔드 튜토리얼

## 개요

이 Notebook에서는 Cedar 정책과 JWT 토큰 클레임을 사용하여 AI Agent의 도구 호출에 세분화된 액세스 제어를 적용하는 **Policy for Amazon Bedrock AgentCore** 구현 방법을 알아봅니다. 정책은 Amazon Bedrock AgentCore Gateway에서 적용됩니다.

### 학습 내용

- JWT 토큰에 사용자 지정 클레임을 추가하도록 Amazon Cognito를 구성하는 방법
- Amazon Bedrock AgentCore Identity를 통해 제공된 JWT 클레임을 검증하는 Cedar 정책을 생성하는 방법
- 속성 기반 액세스 제어(ABAC) 패턴을 구현하는 방법
- Amazon Bedrock AgentCore Gateway를 연결하고 다양한 클레임 시나리오에서 정책 적용을 테스트하고 확인하는 방법

### 주요 개념

**Policy for Amazon Bedrock AgentCore**: JWT 토큰 클레임을 principal 속성으로 사용하여 세분화된 정책에 따라 액세스 요청을 평가하는 Cedar 기반 Policy Engine입니다.

**Amazon Bedrock AgentCore Identity**: Amazon Cognito 또는 OAuth를 지원하는 다른 Identity Provider(IdP)와 통합하여 요청을 인증하고 정책 평가에 사용할 JWT 클레임을 추출합니다.

**Amazon Bedrock AgentCore Gateway**: 개발자가 MCP를 사용하여 대규모로 도구를 구축, 배포, 검색 및 연결할 수 있는 간편하고 안전한 방법을 제공합니다.

**[Cedar 정책](https://www.cedarpolicy.com/en)**: 선택적 조건과 함께 누가(principal), 어떤 리소스에, 무엇(action)을 수행할 수 있는지 정의하는 선언적 정책입니다. 자세한 내용은 [cedarpolicy.com](https://www.cedarpolicy.com/en)에서 확인할 수 있습니다.

### 샘플 아키텍처

```
                                ┌───────────────────────┐
                                │  Policy for AgentCore │
                                │  (Cedar 정책)         │
                                │                       │
                                │  평가 항목:            │
                                │  - principal 태그     │
                                │  - context.input      │
                                │  - resource           │
                                └───────────┬───────────┘
                                            │ 연결됨
                                            ▼
┌─────────────────┐             ┌───────────────────────┐             ┌─────────────┐
│   Amazon        │  JWT 토큰   │  Amazon Bedrock       │             │   Lambda    │
│   Cognito       │────────────▶│  AgentCore Gateway    │────────────▶│   대상      │
│   + AWS Lambda  │  클레임     │                       │  ALLOWED 시 │   (도구)    │
└─────────────────┘  포함       └───────────────────────┘             └─────────────┘
```

### 사전 요구 사항

- 적절한 IAM 권한이 있는 AWS 계정
- OAuth 권한 부여자가 이미 구성된 Amazon Bedrock AgentCore Gateway
- 앱 클라이언트(M2M)가 있는 Amazon Cognito User Pool
- boto3 및 requests가 설치된 Python 3.8 이상
- 구성된 AWS 자격 증명

---

## Part 1: 설정 및 구성

먼저 필요한 종속성을 설치하고 구성을 초기화합니다.

In [ ]:
# 필수 패키지 설치
%pip install -r requirements.txt

In [ ]:
import json
import os
import time
import base64
import zipfile
import tempfile
from pathlib import Path
from typing import Dict, Any, Optional, List

import boto3
import requests
from botocore.exceptions import ClientError

print("✓ Libraries imported successfully")

### Step 1.1: 구성 불러오기 또는 생성

이 튜토리얼에는 Amazon Bedrock AgentCore Gateway 세부 정보와 Amazon Cognito 클라이언트 정보가 포함된 `gateway_config.json` 파일이 필요합니다. 이러한 리소스가 아직 없다면 같은 폴더에 있는 유틸리티 스크립트 `setup-gateway.py`를 실행하여 생성할 수 있습니다.

#### 예상 구성 구조

```json
{
  "gateway_url": "https://<gateway-id>.gateway.policy-registry.<region>.amazonaws.com/mcp",
  "gateway_id": "<gateway-id>",
  "gateway_arn": "arn:aws:policy-registry:<region>:<account-id>:gateway/<gateway-id>",
  "region": "<region>",
  "client_info": {
    "client_id": "<cognito-app-client-id>",
    "client_secret": "<cognito-app-client-secret>",
    "user_pool_id": "<region>_<pool-id>",
    "token_endpoint": "https://<domain>.auth.<region>.amazoncognito.com/oauth2/token"
  },
  "policy_engine_id": "<optional-policy-engine-id>"
}
```

#### 사전 요구 사항 설정

이러한 리소스가 아직 구성되지 않았다면 이 Notebook에서 직접 설정 스크립트를 실행할 수 있습니다. 스크립트는 다음 작업을 수행합니다.
- OAuth 권한 부여를 사용하는 Amazon Bedrock AgentCore Gateway 생성
- 테스트용 샘플 Refund Lambda 함수 생성
- Lambda를 Gateway의 대상으로 연결
- 구성을 `gateway_config.json`에 저장

아래 셀을 실행하여 설정 스크립트를 실행하세요.


In [ ]:
# 사용자에게 AWS 리전 입력 요청
session = boto3.Session()
SETUP_REGION = session.region_name
if not SETUP_REGION:
    SETUP_REGION = input("Enter AWS region (e.g., us-east-1, us-west-2): ").strip()
    if not SETUP_REGION:
        raise ValueError("AWS region is required")

print(f"Using region: {SETUP_REGION}")

# 선택 사항: 신뢰 관계가 구성된 IAM 역할이 있으면 해당 ARN 지정
# 새 역할을 자동으로 생성하려면 None으로 유지
SETUP_ROLE_ARN = None  # 예: "arn:aws:iam::123456789012:role/MyGatewayRole"

In [ ]:
# 이 셀을 실행하여 Gateway와 Lambda 대상을 자동으로 설정
import subprocess
import sys

cmd = [sys.executable, "setup-gateway.py", "--region", SETUP_REGION]
if SETUP_ROLE_ARN:
    cmd.extend(["--role-arn", SETUP_ROLE_ARN])

result = subprocess.run(cmd, capture_output=False, text=True)
if result.returncode != 0:
    print(f"Setup failed with return code: {result.returncode}")

또는 다음 가이드에 따라 리소스를 수동으로 설정할 수 있습니다.

1. **Amazon Bedrock AgentCore Gateway**: [Gateway 빠른 시작 가이드](https://docs.aws.amazon.com/policy-registry/latest/devguide/gateway.html)
2. **Gateway에 대상 추가**: [Gateway 대상 문서](https://docs.aws.amazon.com/policy-registry/latest/devguide/gateway-targets.html)
3. **Amazon Cognito User Pool**: [Amazon Cognito 개발자 가이드](https://docs.aws.amazon.com/cognito/latest/developerguide/cognito-user-pools.html)


In [ ]:
# gateway_config.json 템플릿
CONFIG_TEMPLATE = {
    "gateway_url": "https://<gateway-id>.gateway.policy-registry.<region>.amazonaws.com/mcp",
    "gateway_id": "<gateway-id>",
    "gateway_arn": "arn:aws:policy-registry:<region>:<account-id>:gateway/<gateway-id>",
    "region": "<region>",
    "client_info": {
        "client_id": "<cognito-app-client-id>",
        "client_secret": "<cognito-app-client-secret>",
        "user_pool_id": "<region>_<pool-id>",
        "token_endpoint": "https://<domain>.auth.<region>.amazoncognito.com/oauth2/token",
    },
}


def load_or_create_gateway_config() -> Dict[str, Any]:
    """
    gateway_config.json에서 Gateway 구성을 불러옵니다.
    파일이 없으면 사용자가 값을 입력할 수 있는 템플릿을 생성합니다.
    """
    config_path = Path.cwd() / "gateway_config.json"

    if not config_path.exists():
        # 템플릿 파일 생성
        with open(config_path, "w", encoding="utf-8") as f:
            json.dump(CONFIG_TEMPLATE, f, indent=2)

        print("⚠️  gateway_config.json not found!")
        print(f"\n✓ Created template at: {config_path}")
        print("\nPlease fill in the configuration with your actual values:")
        print("  1. Set up an Amazon Bedrock AgentCore Gateway")
        print("  2. Create an Amazon Cognito User Pool with an app client (M2M)")
        print("  3. Update gateway_config.json with your resource details")
        print("  4. Re-run this cell")
        print("\nDocumentation:")
        print("  - Gateway: https://docs.aws.amazon.com/policy-registry/latest/devguide/gateway.html")
        print("  - Cognito: https://docs.aws.amazon.com/cognito/latest/developerguide/cognito-user-pools.html")
        raise FileNotFoundError("Please configure gateway_config.json and re-run this cell.")

    with open(config_path, "r", encoding="utf-8") as f:
        config = json.load(f)

    # 필수 필드 검증
    required_fields = [
        "gateway_url",
        "gateway_id",
        "gateway_arn",
        "region",
        "client_info",
    ]
    missing = [f for f in required_fields if f not in config or "<" in str(config.get(f, ""))]

    if missing:
        print("⚠️  Configuration incomplete!")
        print(f"   Please update these fields in gateway_config.json: {missing}")
        raise ValueError(f"Missing or placeholder values in config: {missing}")

    # client_info 필드 검증
    client_info_fields = [
        "client_id",
        "client_secret",
        "user_pool_id",
        "token_endpoint",
    ]
    client_info = config.get("client_info", {})
    missing_client = [f for f in client_info_fields if f not in client_info or "<" in str(client_info.get(f, ""))]

    if missing_client:
        print("⚠️  Client info incomplete!")
        print(f"   Please update client_info fields: {missing_client}")
        raise ValueError(f"Missing or placeholder values in client_info: {missing_client}")

    return config


# 구성 불러오기
CONFIG = load_or_create_gateway_config()

# 주요 값 추출
REGION = CONFIG["region"]
GATEWAY_URL = CONFIG["gateway_url"]
GATEWAY_ID = CONFIG["gateway_id"]
GATEWAY_ARN = CONFIG["gateway_arn"]
USER_POOL_ID = CONFIG["client_info"]["user_pool_id"]
CLIENT_ID = CONFIG["client_info"]["client_id"]
CLIENT_SECRET = CONFIG["client_info"]["client_secret"]
TOKEN_ENDPOINT = CONFIG["client_info"]["token_endpoint"]
POLICY_ENGINE_ID = CONFIG.get("policy_engine_id")

print("✓ Configuration loaded successfully")
print(f"  Region: {REGION}")
print(f"  Gateway ID: {GATEWAY_ID}")
print(f"  Gateway URL: {GATEWAY_URL}")
print(f"  User Pool ID: {USER_POOL_ID}")
print(f"  Policy Engine ID: {POLICY_ENGINE_ID or 'Not configured yet'}")

### Step 1.2: AWS 클라이언트 초기화

이 튜토리얼에 필요한 boto3 클라이언트를 생성합니다.

In [ ]:
# AWS 클라이언트 초기화
session = boto3.Session(region_name=REGION)

lambda_client = session.client("lambda")
cognito_client = session.client("cognito-idp")
iam_client = session.client("iam")
sts_client = session.client("sts")

# Policy Engine 및 Cedar 정책을 관리하는 AgentCore Control 클라이언트
policy_client = session.client("bedrock-agentcore-control", region_name=REGION)

# 현재 계정 정보 가져오기
ACCOUNT_ID = sts_client.get_caller_identity()["Account"]

print("✓ AWS clients initialized")
print(f"  Account ID: {ACCOUNT_ID}")
print(f"  Region: {REGION}")

### Step 1.3: Gateway 권한 부여자 구성 검증

이 단계에서는 Gateway의 JWT 권한 부여자가 Cognito 액세스 토큰에 맞게 구성되었는지 확인합니다.

**중요**: Amazon Cognito 액세스 토큰에는 `aud`(audience) 클레임이 포함되지 않습니다. Gateway에 `allowedAudience`가 구성되어 있으면 토큰 검증이 401 오류와 함께 실패합니다. 이 단계에서는 구성을 확인하고 필요한 경우 수정합니다.

In [ ]:
# Gateway 구성을 관리하는 Gateway Control 클라이언트
gateway_control_client = session.client("bedrock-agentcore-control", region_name=REGION)


def get_gateway_details() -> Dict[str, Any]:
    """현재 Gateway의 세부 정보를 가져옵니다."""
    return gateway_control_client.get_gateway(gatewayIdentifier=GATEWAY_ID)


def wait_for_gateway_ready(max_wait: int = 300, poll_interval: int = 5) -> bool:
    """Gateway가 READY 상태가 될 때까지 기다립니다."""
    terminal_states = {"READY", "FAILED", "UPDATE_UNSUCCESSFUL"}
    start_time = time.time()

    while time.time() - start_time < max_wait:
        gateway = get_gateway_details()
        status = gateway.get("status", "UNKNOWN")
        print(f"  Gateway status: {status}")

        if status == "READY":
            return True
        if status in terminal_states:
            print(f"  ✗ Gateway reached terminal state: {status}")
            return False

        time.sleep(poll_interval)

    print("  ✗ Timeout waiting for gateway")
    return False


def validate_and_fix_gateway_authorizer() -> bool:
    """
    Gateway 권한 부여자 구성을 검증하고 필요한 경우 수정합니다.

    Cognito 액세스 토큰에는 'aud' 클레임이 없으므로 allowedAudience를 설정하면
    Gateway가 유효한 토큰을 거부합니다.

    반환값:
        구성이 유효하거나 성공적으로 수정되었으면 True
    """
    print("\nValidating Gateway Authorizer Configuration")
    print("=" * 70)

    gw = get_gateway_details()
    jwt_config = gw.get("authorizerConfiguration", {}).get("customJWTAuthorizer", {})

    # 현재 구성 확인
    discovery_url = jwt_config.get("discoveryUrl")
    allowed_clients = jwt_config.get("allowedClients", [])
    allowed_audience = jwt_config.get("allowedAudience", [])
    allowed_scopes = jwt_config.get("allowedScopes", [])

    print(f"  Discovery URL: {discovery_url or 'NOT SET'}")
    print(f"  Allowed Clients: {allowed_clients}")
    print(f"  Allowed Audience: {allowed_audience}")
    print(f"  Allowed Scopes: {allowed_scopes}")

    # 예상 discovery URL 구성
    expected_discovery_url = (
        f"https://cognito-idp.{REGION}.amazonaws.com/{USER_POOL_ID}/.well-known/openid-configuration"
    )

    # 구성을 수정해야 하는지 확인
    needs_fix = False
    reasons = []

    if not discovery_url:
        needs_fix = True
        reasons.append("Discovery URL not set")
    elif discovery_url != expected_discovery_url:
        needs_fix = True
        reasons.append("Discovery URL mismatch")

    if CLIENT_ID not in allowed_clients:
        needs_fix = True
        reasons.append(f"Client ID {CLIENT_ID} not in allowed clients")

    # Cognito 액세스 토큰에는 'aud' 클레임이 없으므로 allowedAudience는 비어 있어야 함
    if allowed_audience:
        needs_fix = True
        reasons.append("allowedAudience is set but Cognito access tokens don't have 'aud' claim")

    if not needs_fix:
        print("\n✓ Gateway authorizer configuration is valid")
        return True

    print("\n⚠️  Configuration needs fixing:")
    for reason in reasons:
        print(f"   - {reason}")

    # 구성 수정
    print("\n⏳ Updating gateway authorizer configuration...")

    # 구성에 scope가 있으면 가져오기
    scope = CONFIG.get("client_info", {}).get("scope", "")

    new_auth_config = {
        "customJWTAuthorizer": {
            "discoveryUrl": expected_discovery_url,
            "allowedClients": [CLIENT_ID],
            # allowedAudience를 설정하지 말 것 - Cognito 액세스 토큰에는 'aud' 클레임이 없음
        }
    }

    # scope가 구성되어 있으면 추가
    if scope:
        new_auth_config["customJWTAuthorizer"]["allowedScopes"] = [scope]

    try:
        # 이미 연결된 경우에만 policyEngineConfiguration 포함
        pe_config = gw.get("policyEngineConfiguration", {})
        update_params = dict(
            gatewayIdentifier=GATEWAY_ID,
            name=gw.get("name"),
            roleArn=gw.get("roleArn"),
            protocolType=gw.get("protocolType", "MCP"),
            authorizerType="CUSTOM_JWT",
            authorizerConfiguration=new_auth_config,
        )
        if pe_config and pe_config.get("arn"):
            update_params["policyEngineConfiguration"] = pe_config

        gateway_control_client.update_gateway(**update_params)

        print("\n⏳ Waiting for gateway to become READY...")
        if wait_for_gateway_ready():
            print("\n✓ Gateway authorizer configuration fixed successfully")
            return True
        else:
            print("\n✗ Gateway did not reach READY state")
            return False

    except ClientError as e:
        print(f"\n✗ Error updating gateway: {e}")
        return False


# Gateway 권한 부여자를 검증하고 필요한 경우 수정
validate_and_fix_gateway_authorizer()

---

## Part 2: 도우미 함수

이 유틸리티 함수는 튜토리얼 전반에서 토큰 관리, API 호출 및 응답 분석에 사용됩니다.

In [ ]:
def get_bearer_token() -> str:
    """
    OAuth2 client credentials flow를 사용해 bearer token을 가져옵니다.

    반환값:
        액세스 토큰 문자열
    """
    # 구성에 scope가 있으면 가져오기
    scope = CONFIG.get("client_info", {}).get("scope", "")

    data = {
        "grant_type": "client_credentials",
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
    }

    # scope가 구성되어 있으면 추가
    if scope:
        data["scope"] = scope

    response = requests.post(
        TOKEN_ENDPOINT,
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        data=data,
    )
    response.raise_for_status()
    return response.json()["access_token"]


def decode_token(access_token: str) -> Dict[str, Any]:
    """
    클레임을 확인할 수 있도록 JWT token을 검증 없이 디코딩합니다.

    인수:
        access_token: JWT 액세스 토큰

    반환값:
        딕셔너리로 디코딩한 token payload
    """
    parts = access_token.split(".")
    if len(parts) != 3:
        raise ValueError("Invalid JWT token format")

    # payload 디코딩(필요한 경우 padding 추가)
    payload_encoded = parts[1]
    padding = 4 - len(payload_encoded) % 4
    if padding != 4:
        payload_encoded += "=" * padding

    return json.loads(base64.urlsafe_b64decode(payload_encoded))


def make_gateway_request(bearer_token: str, tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
    """
    Amazon Bedrock AgentCore Gateway에 JSON-RPC 요청을 보냅니다.

    인수:
        bearer_token: OAuth2 액세스 토큰
        tool_name: 호출할 도구 이름
        arguments: 도구 인수

    반환값:
        JSON-RPC 응답
    """
    payload = {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "tools/call",
        "params": {"name": tool_name, "arguments": arguments},
    }

    response = requests.post(
        GATEWAY_URL,
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {bearer_token}",
            "Accept": "application/json",
        },
        json=payload,
    )
    response.raise_for_status()
    return response.json()


def analyze_response(result: Dict[str, Any]) -> str:
    """
    Gateway 응답을 분석해 결과를 판정합니다.

    반환값:
        'ALLOWED', 'DENIED' 또는 'ERROR'
    """
    # JSON-RPC 오류 확인(정책 거부는 특정 메시지가 포함된 오류로 반환됨)
    if "error" in result:
        error_msg = result["error"].get("message", "").lower()
        # 정책 거부는 'tool call not allowed', 'access denied' 등의 오류로 반환됨
        if any(phrase in error_msg for phrase in ["not allowed", "denied", "forbidden", "unauthorized action"]):
            return "DENIED"
        return "ERROR"

    if "result" in result:
        # 결과가 오류를 나타내는지 확인(일부 거부는 이 형식으로 반환됨)
        if result["result"].get("isError", False):
            content = result["result"].get("content", [])
            if content:
                text = content[0].get("text", "").lower() if isinstance(content[0], dict) else str(content[0]).lower()
                if any(phrase in text for phrase in ["not allowed", "denied", "forbidden"]):
                    return "DENIED"
            return "DENIED"
        return "ALLOWED"

    return "UNKNOWN"


def display_test_result(expected: str, actual: str, description: str) -> bool:
    """
    테스트 결과를 서식에 맞춰 표시합니다.

    반환값:
        테스트를 통과했으면 True, 아니면 False
    """
    passed = expected == actual
    status = "✓ PASS" if passed else "✗ FAIL"
    print(f"\n{status}: {description}")
    print(f"   Expected: {expected}")
    print(f"   Actual: {actual}")
    return passed


print("✓ Helper functions defined")

---

## Part 3: Amazon Cognito Lambda 트리거 구성

JWT 토큰에 사용자 지정 클레임을 추가하려면 Amazon Cognito에서 Pre Token Generation AWS Lambda 트리거를 구성해야 합니다.

### 중요 사항

- M2M(machine-to-machine) 클라이언트 자격 증명 흐름에서는 AWS Lambda 트리거 버전 **V3_0**을 **반드시** 사용해야 합니다.
- V3_0을 사용하려면 Amazon Cognito **Essentials** 또는 **Plus** 티어가 필요합니다.
- AWS Lambda 함수는 `department_name`, `groups` 등의 사용자 지정 클레임을 JWT 토큰에 추가합니다.
- 이러한 클레임은 Cedar 정책에서 **principal 태그**가 됩니다.

In [ ]:
def create_lambda_function(claims: Dict[str, Any], function_name: Optional[str] = None) -> str:
    """
    Pre-token generation trigger용 Lambda 함수를 생성하거나 업데이트합니다.

    인수:
        claims: Token에 추가할 claim dictionary
        function_name: 선택적인 사용자 지정 함수 이름

    반환값:
        Lambda 함수 ARN
    """
    if function_name is None:
        function_name = f"cognito-custom-claims-{USER_POOL_ID}"

    print(f"\nConfiguring Lambda Function: {function_name}")
    print("=" * 70)

    # 지정된 클레임으로 Lambda 코드 생성
    claims_json = json.dumps(claims, indent=12)

    lambda_code = f'''
import json

def lambda_handler(event, context):
    """
    Cognito용 pre-token generation V3 Lambda trigger입니다.
    client_credentials를 포함한 모든 흐름의 JWT token에 사용자 지정 claim을 추가합니다.
    """
    print(f"Event: {{json.dumps(event)}}")
    print(f"Trigger Source: {{event.get('triggerSource', 'unknown')}}")
    
    # Token에 사용자 지정 claim 추가
    event['response'] = {{
        'claimsAndScopeOverrideDetails': {{
            'accessTokenGeneration': {{
                'claimsToAddOrOverride': {claims_json}
            }},
            'idTokenGeneration': {{
                'claimsToAddOrOverride': {claims_json}
            }}
        }}
    }}
    
    print(f"Modified event: {{json.dumps(event)}}")
    return event
'''

    # 배포 패키지 생성
    with tempfile.NamedTemporaryFile(suffix=".zip", delete=False) as tmp_file:
        zip_path = tmp_file.name
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
            zipf.writestr("lambda_function.py", lambda_code)

    try:
        with open(zip_path, "rb") as f:
            zip_content = f.read()

        # 기존 함수 업데이트 시도
        try:
            lambda_client.update_function_code(FunctionName=function_name, ZipFile=zip_content)
            print("✓ Updated Lambda function code")
            response = lambda_client.get_function(FunctionName=function_name)
            return response["Configuration"]["FunctionArn"]

        except lambda_client.exceptions.ResourceNotFoundException:
            # IAM 역할을 사용하여 새 함수 생성
            role_name = f"{function_name}-role"
            role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{role_name}"

            # 필요한 경우 IAM 역할 생성
            try:
                iam_client.create_role(
                    RoleName=role_name,
                    AssumeRolePolicyDocument=json.dumps(
                        {
                            "Version": "2012-10-17",
                            "Statement": [
                                {
                                    "Effect": "Allow",
                                    "Principal": {"Service": "lambda.amazonaws.com"},
                                    "Action": "sts:AssumeRole",
                                }
                            ],
                        }
                    ),
                )
                iam_client.attach_role_policy(
                    RoleName=role_name,
                    PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
                )
                print(f"✓ Created IAM role: {role_name}")
                print("  Waiting for IAM role propagation...")
                time.sleep(10)
            except iam_client.exceptions.EntityAlreadyExistsException:
                print(f"  IAM role already exists: {role_name}")

            response = lambda_client.create_function(
                FunctionName=function_name,
                Runtime="python3.12",
                Role=role_arn,
                Handler="lambda_function.lambda_handler",
                Code={"ZipFile": zip_content},
                Timeout=30,
                MemorySize=128,
            )
            print("✓ Created Lambda function")
            return response["FunctionArn"]
    finally:
        os.remove(zip_path)


print("✓ Lambda creation function defined")

In [ ]:
def configure_cognito_trigger(lambda_arn: str) -> None:
    """
    Cognito User Pool에 Lambda trigger V3_0을 구성합니다.

    인수:
        lambda_arn: Lambda 함수 ARN
    """
    print("\nConfiguring Cognito User Pool Trigger")
    print("=" * 70)

    # V3_0 트리거로 User Pool 업데이트(M2M에 필요)
    cognito_client.update_user_pool(
        UserPoolId=USER_POOL_ID,
        LambdaConfig={
            "PreTokenGenerationConfig": {
                "LambdaVersion": "V3_0",
                "LambdaArn": lambda_arn,
            }
        },
    )
    print("✓ User Pool trigger configured (V3_0)")
    print(f"  User Pool ID: {USER_POOL_ID}")
    print(f"  Lambda ARN: {lambda_arn}")

    # Cognito에 Lambda 권한 추가
    try:
        lambda_client.add_permission(
            FunctionName=lambda_arn,
            StatementId=f"CognitoInvoke-{USER_POOL_ID}",
            Action="lambda:InvokeFunction",
            Principal="cognito-idp.amazonaws.com",
            SourceArn=f"arn:aws:cognito-idp:{REGION}:{ACCOUNT_ID}:userpool/{USER_POOL_ID}",
        )
        print("✓ Lambda permission added for Cognito")
    except lambda_client.exceptions.ResourceConflictException:
        print("  Lambda permission already exists")

    print("\n⚠️  IMPORTANT: V3_0 trigger requires Cognito Essentials or Plus tier")


print("✓ Cognito trigger function defined")

---

## Part 4: Policy Engine 함수

이 함수는 Policy for Amazon Bedrock AgentCore 서비스와 상호 작용하여 Cedar 정책을 생성하고 관리합니다.

### JWT 클레임용 Cedar 정책 구문

Cedar 정책에서는 **principal 태그**를 통해 JWT 클레임에 액세스합니다.

| 패턴 | Cedar 구문 |
|---------|-------------|
| 클레임 존재 여부 확인 | `principal.hasTag("claim_name")` |
| 정확히 일치 | `principal.getTag("claim_name") == "value"` |
| 패턴 일치 | `principal.getTag("claim_name") like "*value*"` |
| 입력값 검증 | `context.input.field <= value` |

In [ ]:
def create_policy_engine(name: str) -> Optional[str]:
    """
    새 Policy Engine을 생성합니다.

    인수:
        name: Policy Engine 이름

    반환값:
        성공하면 Policy Engine ID, 아니면 None
    """
    print(f"\nCreating Policy Engine: {name}")
    print("=" * 70)

    try:
        import uuid

        response = policy_client.create_policy_engine(
            name=name,
            description=f"Policy engine created at {time.strftime('%Y-%m-%d %H:%M:%S')}",
            clientToken=str(uuid.uuid4()),
        )

        policy_engine_id = response["policyEngineId"]
        print("✓ Policy engine created")
        print(f"  Policy Engine ID: {policy_engine_id}")

        return policy_engine_id

    except ClientError as e:
        print(f"✗ Error creating policy engine: {e}")
        return None


def get_policy_engine(policy_engine_id: str) -> Optional[Dict[str, Any]]:
    """
    Policy Engine의 세부 정보를 가져옵니다.
    """
    try:
        return policy_client.get_policy_engine(policyEngineId=policy_engine_id)
    except ClientError:
        return None


def wait_for_policy_engine_active(policy_engine_id: str, timeout: int = 300) -> bool:
    """
    Policy Engine이 ACTIVE 상태가 될 때까지 기다립니다.

    인수:
        policy_engine_id: Policy Engine ID
        timeout: 최대 대기 시간(초)

    반환값:
        ACTIVE 상태이면 True, 시간 초과 또는 실패이면 False
    """
    print("\nWaiting for Policy Engine to become ACTIVE...")
    start_time = time.time()

    while time.time() - start_time < timeout:
        engine = get_policy_engine(policy_engine_id)
        if not engine:
            time.sleep(5)
            continue

        status = engine.get("status")
        print(f"  Status: {status}")

        if status == "ACTIVE":
            print("✓ Policy engine is ACTIVE")
            return True

        if status in ["CREATE_FAILED", "UPDATE_FAILED", "DELETE_FAILED"]:
            print(f"✗ Policy engine failed: {engine.get('statusReason', 'Unknown')}")
            return False

        time.sleep(5)

    print("✗ Timeout waiting for policy engine")
    return False


print("✓ Policy engine functions defined")

In [ ]:
def create_cedar_policy(policy_name: str, cedar_statement: str, description: str = "") -> Optional[str]:
    """
    Policy Engine에 Cedar 정책을 생성합니다.

    인수:
        policy_name: 고유한 정책 이름
        cedar_statement: Cedar 정책문
        description: 정책 설명

    반환값:
        성공하면 정책 ID, 아니면 None
    """
    print(f"\nCreating Cedar Policy: {policy_name}")
    print("=" * 70)
    print("\nCedar Statement:")
    print("-" * 60)
    print(cedar_statement)
    print("-" * 60)

    try:
        response = policy_client.create_policy(
            policyEngineId=POLICY_ENGINE_ID,
            name=policy_name,
            description=description or f"Policy: {policy_name}",
            definition={"cedar": {"statement": cedar_statement}},
        )

        policy_id = response["policyId"]
        policy_status = response["status"]

        print("\n✓ Policy created successfully")
        print(f"  Policy ID: {policy_id}")
        print(f"  Status: {policy_status}")

        return policy_id

    except ClientError as e:
        error_code = e.response["Error"]["Code"]
        error_msg = e.response["Error"]["Message"]
        print(f"\n✗ Error creating policy: {error_code}")
        print(f"  {error_msg}")
        return None


def get_policy(policy_id: str) -> Optional[Dict[str, Any]]:
    """
    상태를 포함한 정책 세부 정보를 가져옵니다.

    인수:
        policy_id: 조회할 정책 ID

    반환값:
        정책 세부 정보 딕셔너리, 찾지 못하면 None
    """
    try:
        return policy_client.get_policy(policyEngineId=POLICY_ENGINE_ID, policyId=policy_id)
    except ClientError:
        return None


def wait_for_policy_active(policy_id: str, timeout: int = 60) -> bool:
    """
    정책이 ACTIVE 상태가 될 때까지 기다립니다.

    인수:
        policy_id: 확인할 정책 ID
        timeout: 최대 대기 시간(초)

    반환값:
        정책이 ACTIVE 상태이면 True, 아니면 False
    """
    start_time = time.time()

    while time.time() - start_time < timeout:
        policy = get_policy(policy_id)
        if not policy:
            print(f"  ⚠️  Policy not found: {policy_id}")
            return False

        status = policy.get("status")
        print(f"  Policy status: {status}")

        if status == "ACTIVE":
            return True

        if status in ["CREATE_FAILED", "UPDATE_FAILED"]:
            print(f"  ✗ Policy failed: {policy.get('statusReason', 'Unknown')}")
            return False

        time.sleep(3)

    print("  ✗ Timeout waiting for policy to become ACTIVE")
    return False


def delete_policy(policy_id: str) -> bool:
    """
    Policy Engine에서 정책을 삭제합니다.
    """
    try:
        policy_client.delete_policy(policyEngineId=POLICY_ENGINE_ID, policyId=policy_id)
        print(f"✓ Deleted policy: {policy_id}")
        return True
    except ClientError as e:
        print(f"⚠️  Could not delete policy {policy_id}: {e}")
        return False


def list_policies() -> List[Dict[str, Any]]:
    """
    Policy Engine의 모든 정책을 나열합니다.
    """
    try:
        response = policy_client.list_policies(policyEngineId=POLICY_ENGINE_ID)
        return response.get("policies", [])
    except ClientError:
        return []


print("✓ Cedar policy functions defined")

### Step 4.1: Policy Engine 존재 확인

Policy Engine이 있는지 확인하고 필요한 경우 새로 생성합니다.

In [ ]:
def ensure_policy_engine() -> str:
    """
    Policy Engine이 존재하며 활성 상태인지 확인합니다.
    필요한 경우 생성하고 gateway_config.json을 업데이트합니다.

    반환값:
        Policy Engine ID
    """
    global POLICY_ENGINE_ID, CONFIG

    print("\nEnsuring Policy Engine Exists")
    print("=" * 70)

    # Policy Engine ID가 이미 있는지 확인
    if POLICY_ENGINE_ID:
        engine = get_policy_engine(POLICY_ENGINE_ID)
        if engine and engine.get("status") == "ACTIVE":
            print(f"✓ Using existing policy engine: {POLICY_ENGINE_ID}")
            return POLICY_ENGINE_ID

    # 기존 Policy Engine 목록 조회
    try:
        response = policy_client.list_policy_engines()
        engines = response.get("policyEngines", [])

        for engine in engines:
            if engine.get("status") == "ACTIVE":
                POLICY_ENGINE_ID = engine["policyEngineId"]
                print(f"✓ Found existing ACTIVE policy engine: {POLICY_ENGINE_ID}")
                break
    except ClientError:
        pass

    # 필요한 경우 새 Policy Engine 생성
    if not POLICY_ENGINE_ID:
        engine_name = f"PolicyEngine_{int(time.time())}"
        POLICY_ENGINE_ID = create_policy_engine(engine_name)

        if not POLICY_ENGINE_ID:
            raise RuntimeError("Failed to create policy engine")

        if not wait_for_policy_engine_active(POLICY_ENGINE_ID):
            raise RuntimeError("Policy engine did not become ACTIVE")

    # gateway_config.json에 저장
    CONFIG["policy_engine_id"] = POLICY_ENGINE_ID
    with open("gateway_config.json", "w") as f:
        json.dump(CONFIG, f, indent=2)
    print("✓ Saved policy_engine_id to gateway_config.json")

    return POLICY_ENGINE_ID


# Policy Engine 존재 확인
POLICY_ENGINE_ID = ensure_policy_engine()

### Step 4.2: Gateway에 Policy Engine 연결

정책을 적용하려면 Policy Engine을 Gateway에 연결해야 합니다. 이 단계에서는 Gateway에 Policy Engine이 이미 구성되어 있는지 확인하고, 구성되어 있지 않으면 연결합니다.

In [ ]:
# 참고: gateway_control_client, get_gateway_details, wait_for_gateway_ready는
# Step 1.3에서 이미 정의됨


def attach_policy_engine_to_gateway(mode: str = "ENFORCE") -> bool:
    """
    Policy Engine이 아직 연결되지 않았다면 Gateway에 연결합니다.
    다른 Policy Engine이 연결되어 있으면 현재 Policy Engine으로 다시 연결합니다.

    인수:
        mode: Policy Engine 모드('LOG_ONLY' 또는 'ENFORCE')

    반환값:
        성공했거나 이미 연결되어 있으면 True, 아니면 False
    """
    print("\nAttaching Policy Engine to Gateway")
    print("=" * 70)

    # 현재 Gateway 구성 가져오기
    gateway_config = get_gateway_details()

    # 올바른 Policy Engine이 이미 연결되어 있는지 확인
    existing_pe = gateway_config.get("policyEngineConfiguration", {})
    engine = get_policy_engine(POLICY_ENGINE_ID)
    if not engine:
        print("✗ Could not get policy engine details")
        return False

    policy_engine_arn = engine.get("policyEngineArn")

    if existing_pe.get("arn") == policy_engine_arn and existing_pe.get("mode") == mode:
        print(f"✓ Correct Policy Engine already attached: {existing_pe.get('arn')}")
        print(f"  Mode: {existing_pe.get('mode', 'N/A')}")
        return True

    if existing_pe.get("arn"):
        print(f"  Replacing attached policy engine: {existing_pe.get('arn')}")
    print(f"  Policy Engine ARN: {policy_engine_arn}")
    print(f"  Mode: {mode}")

    try:
        # Gateway 업데이트 시 기존 권한 부여자 구성을 유지해야 함
        auth_config = gateway_config.get("authorizerConfiguration", {})

        gateway_control_client.update_gateway(
            gatewayIdentifier=GATEWAY_ID,
            name=gateway_config.get("name"),
            roleArn=gateway_config.get("roleArn"),
            protocolType=gateway_config.get("protocolType", "MCP"),
            authorizerType=gateway_config.get("authorizerType", "CUSTOM_JWT"),
            authorizerConfiguration=auth_config,
            policyEngineConfiguration={"arn": policy_engine_arn, "mode": mode},
        )

        print("✓ Gateway update request accepted")
        print("\n⏳ Waiting for gateway to become READY...")

        if wait_for_gateway_ready():
            print("✓ Policy Engine attached successfully")
            return True
        else:
            print("✗ Gateway did not reach READY state")
            return False

    except ClientError as e:
        print(f"✗ Error updating gateway: {e}")
        return False


# Gateway에 Policy Engine 연결
attach_policy_engine_to_gateway(mode="ENFORCE")

---

## Part 5: 테스트 시나리오 1 - 부서 기반 액세스 제어

이 시나리오에서는 **finance** 부서 사용자의 요청만 허용하는 정책을 생성합니다.

### Cedar 정책 패턴

```cedar
permit(principal, action, resource)
when {
    principal.hasTag("department_name") &&
    principal.getTag("department_name") == "finance"
};
```

### Step 5.0: 기존 정책 정리(선택 사항)

새 정책을 생성하기 전에 기존 정책을 모두 삭제하여 테스트 환경을 정리하는 것이 좋습니다. 이렇게 하면 이전 정책과 새 정책 간의 충돌을 방지할 수 있습니다.

In [ ]:
def cleanup_existing_policies(require_confirmation: bool = True) -> int:
    """
    Policy Engine의 기존 정책을 모두 삭제합니다.

    인수:
        require_confirmation: True이면 삭제 전에 사용자에게 확인을 요청함

    반환값:
        삭제한 정책 수
    """
    print("\n🧹 Checking for existing policies...")
    print("=" * 70)

    policies = list_policies()

    if not policies:
        print("✓ No existing policies found. Ready to proceed.")
        return 0

    print(f"\n⚠️  Found {len(policies)} existing policy/policies:")
    for p in policies:
        print(f"   - {p.get('name', 'unnamed')} (ID: {p.get('policyId')}, Status: {p.get('status')})")

    if require_confirmation:
        print("\n" + "-" * 70)
        confirm = input("Do you want to DELETE all existing policies? (yes/no): ").strip().lower()
        if confirm != "yes":
            print("\n⏭️  Skipping cleanup. Existing policies will remain.")
            print("   Note: This may cause unexpected policy evaluation results.")
            return 0

    print("\n🗑️  Deleting existing policies...")
    deleted_count = 0
    for p in policies:
        policy_id = p.get("policyId")
        if policy_id and delete_policy(policy_id):
            deleted_count += 1

    print(f"\n✓ Deleted {deleted_count}/{len(policies)} policies")
    return deleted_count


# 마지막 정리를 위해 생성된 정책 추적
CREATED_POLICIES = []

# 테스트 시작 전에 기존 정책 정리(확인 불필요)
cleanup_existing_policies(require_confirmation=False)

### Step 5.1: Finance 부서 클레임으로 Lambda 구성

In [ ]:
print("=" * 70)
print("TEST SCENARIO 1: Department-Based Access Control")
print("=" * 70)

# department_name = "finance"로 Lambda 구성
claims_finance = {
    "department_name": "finance",
    "employee_level": "senior",
    "cost_center": "CC-1001",
}

lambda_arn = create_lambda_function(claims_finance)
configure_cognito_trigger(lambda_arn)

# 토큰을 요청하기 전에 Lambda 트리거가 전파될 때까지 대기
print("\n⏳ Waiting for Cognito trigger to propagate...")
time.sleep(15)

print("\n✓ Lambda configured with claims:")
print(json.dumps(claims_finance, indent=2))

### Step 5.2: 토큰에 사용자 지정 클레임이 포함되어 있는지 확인

In [ ]:
print("\nVerifying Token Claims")
print("=" * 70)

token = get_bearer_token()
claims = decode_token(token)

print("\nToken Claims (relevant):")
print(f"  department_name: {claims.get('department_name', 'NOT PRESENT')}")
print(f"  employee_level: {claims.get('employee_level', 'NOT PRESENT')}")
print(f"  cost_center: {claims.get('cost_center', 'NOT PRESENT')}")
print(f"  client_id: {claims.get('client_id', 'NOT PRESENT')}")

if claims.get("department_name") == "finance":
    print("\n✓ Custom claims verified in token")
else:
    print("\n⚠️  Custom claims not found - Lambda trigger may not be configured correctly")

### Step 5.3: 부서 검증용 Cedar 정책 생성

In [ ]:
policy_name = f"dept_policy_{int(time.time())}"

cedar_statement = f'''permit(principal,
    action == AgentCore::Action::"RefundToolTarget___refund",
    resource == AgentCore::Gateway::"{GATEWAY_ARN}")
when {{
    principal.hasTag("department_name") &&
    principal.getTag("department_name") == "finance"
}};'''

print(f"Cedar statement:\n{cedar_statement}")

policy_id = create_cedar_policy(
    policy_name=policy_name,
    cedar_statement=cedar_statement,
    description="Allow requests only from finance department",
)

if policy_id:
    CREATED_POLICIES.append(policy_id)

    # 정책이 ACTIVE 상태가 될 때까지 대기(테스트 전에 필요)
    print("\n⏳ Waiting for policy to become ACTIVE...")
    if wait_for_policy_active(policy_id):
        print("✓ Policy is ACTIVE and ready for testing")
    else:
        print("\n⚠️  Policy did not become ACTIVE. Tests may fail.")
        print("   Check the policy status in the AWS Console.")
else:
    print("\n✗ Failed to create policy. Cannot proceed with tests.")

### Step 5.4: Finance 부서로 테스트(예상 결과: ALLOWED)

In [ ]:
print("\n" + "=" * 70)
print("Test 1.1: Request with department_name='finance'")
print("=" * 70)

token = get_bearer_token()
result = make_gateway_request(
    bearer_token=token,
    tool_name="RefundToolTarget___refund",
    arguments={"amount": 500, "orderId": "test-dept-finance"},
)

print("\nRequest: RefundToolTarget___refund(amount=500)")
print("\nResponse:")
print(json.dumps(result, indent=2))

outcome = analyze_response(result)
display_test_result("ALLOWED", outcome, "Finance department should be ALLOWED")

### Step 5.5: Engineering 부서로 테스트(예상 결과: DENIED)

In [ ]:
print("\n" + "=" * 70)
print("Test 1.2: Request with department_name='engineering'")
print("=" * 70)

# 다른 부서로 Lambda 업데이트
claims_engineering = {
    "department_name": "engineering",
    "employee_level": "senior",
    "cost_center": "CC-2001",
}

lambda_arn = create_lambda_function(claims_engineering)
print("\n✓ Lambda updated with department_name='engineering'")

# Lambda 변경 사항이 전파될 때까지 대기
print("\n⏳ Waiting for Lambda changes to propagate...")
time.sleep(5)

# 새 토큰을 가져와 테스트
token = get_bearer_token()
claims = decode_token(token)
print(f"\nToken department_name: {claims.get('department_name')}")

result = make_gateway_request(
    bearer_token=token,
    tool_name="RefundToolTarget___refund",
    arguments={"amount": 500, "orderId": "test-dept-engineering"},
)

print("\nRequest: RefundToolTarget___refund(amount=500)")
print("\nResponse:")
print(json.dumps(result, indent=2))

outcome = analyze_response(result)
display_test_result("DENIED", outcome, "Engineering department should be DENIED")

---

## Part 6: 테스트 시나리오 2 - 그룹 기반 액세스 제어

이 시나리오에서는 **admins** 그룹에 속한 사용자의 요청만 허용하는 정책을 생성합니다.

### Cedar 정책 패턴

토큰에서 groups가 문자열로 직렬화되므로 패턴 일치에 `like` 연산자를 사용합니다.

```cedar
permit(principal, action, resource)
when {
    principal.hasTag("groups") &&
    principal.getTag("groups") like "*admins*"
};
```

In [ ]:
# 이전 정책 정리
print("=" * 70)
print("TEST SCENARIO 2: Groups-Based Access Control")
print("=" * 70)

print("\nCleaning up previous policies...")
for pid in CREATED_POLICIES:
    delete_policy(pid)
CREATED_POLICIES.clear()

### Step 6.1: Groups 클레임으로 Lambda 구성

In [ ]:
# "admins"를 포함하는 groups로 Lambda 구성
claims_with_admins = {
    "groups": ["admins", "developers", "team-alpha"],
    "department_name": "finance",
    "employee_level": "senior",
}

lambda_arn = create_lambda_function(claims_with_admins)
configure_cognito_trigger(lambda_arn)

print("\n✓ Lambda configured with groups:")
print(f"   groups: {claims_with_admins['groups']}")

### Step 6.2: 토큰에 Groups 클레임이 포함되어 있는지 확인

In [ ]:
print("\nVerifying Token Claims")
print("=" * 70)

token = get_bearer_token()
claims = decode_token(token)

print("\nToken Claims (relevant):")
print(f"  groups: {claims.get('groups', 'NOT PRESENT')}")
print(f"  department_name: {claims.get('department_name', 'NOT PRESENT')}")

### Step 6.3: Groups 검증용 Cedar 정책 생성

In [ ]:
policy_name = f"groups_policy_{int(time.time())}"

cedar_statement = f'''permit(principal,
    action == AgentCore::Action::"RefundToolTarget___refund",
    resource == AgentCore::Gateway::"{GATEWAY_ARN}")
when {{
    principal.hasTag("groups") &&
    principal.getTag("groups") like "*admins*"
}};'''

print(f"Cedar statement:\n{cedar_statement}")

policy_id = create_cedar_policy(
    policy_name=policy_name,
    cedar_statement=cedar_statement,
    description="Allow requests only from users in admins group",
)

if policy_id:
    CREATED_POLICIES.append(policy_id)

    # 정책이 ACTIVE 상태가 될 때까지 대기(테스트 전에 필요)
    print("\n⏳ Waiting for policy to become ACTIVE...")
    if wait_for_policy_active(policy_id):
        print("✓ Policy is ACTIVE and ready for testing")
    else:
        print("\n⚠️  Policy did not become ACTIVE. Tests may fail.")
else:
    print("\n✗ Failed to create policy. Cannot proceed with tests.")

### Step 6.4: Admins 그룹으로 테스트(예상 결과: ALLOWED)

In [ ]:
print("\n" + "=" * 70)
print("Test 2.1: Request with groups=['admins', 'developers', 'team-alpha']")
print("=" * 70)

token = get_bearer_token()
result = make_gateway_request(
    bearer_token=token,
    tool_name="RefundToolTarget___refund",
    arguments={"amount": 500, "orderId": "test-groups-admins"},
)

print("\nRequest: RefundToolTarget___refund(amount=500)")
print("\nResponse:")
print(json.dumps(result, indent=2))

outcome = analyze_response(result)
display_test_result("ALLOWED", outcome, "User with 'admins' group should be ALLOWED")

### Step 6.5: Admins 그룹 없이 테스트(예상 결과: DENIED)

In [ ]:
print("\n" + "=" * 70)
print("Test 2.2: Request with groups=['developers', 'team-alpha']")
print("=" * 70)

# admins 그룹 없이 Lambda 업데이트
claims_no_admins = {
    "groups": ["developers", "team-alpha"],
    "department_name": "finance",
    "employee_level": "senior",
}

lambda_arn = create_lambda_function(claims_no_admins)
print("\n✓ Lambda updated with groups (no admins):")

# Lambda 변경 사항이 전파될 때까지 대기
print("\n⏳ Waiting for Lambda changes to propagate...")
time.sleep(5)
print(f"   groups: {claims_no_admins['groups']}")

# 새 토큰을 가져와 테스트
token = get_bearer_token()
claims = decode_token(token)
print(f"\nToken groups: {claims.get('groups')}")

result = make_gateway_request(
    bearer_token=token,
    tool_name="RefundToolTarget___refund",
    arguments={"amount": 500, "orderId": "test-groups-no-admins"},
)

print("\nRequest: RefundToolTarget___refund(amount=500)")
print("\nResponse:")
print(json.dumps(result, indent=2))

outcome = analyze_response(result)
display_test_result("DENIED", outcome, "User without 'admins' group should be DENIED")

---

## Part 7: 테스트 시나리오 3 - Principal ID 기반 액세스 제어

이 시나리오에서는 Client Credentials Flow의 `sub` 클레임으로 식별되는 특정 principal의 요청만 허용하는 정책을 생성합니다.

### Cedar 정책 패턴

Client Credentials Flow에서 JWT 토큰의 `sub` 클레임은 `client_id`와 같습니다. 모든 JWT 클레임은 **principal 태그**로 제공되므로 `getTag("sub")`를 통해 호출자의 ID가 일치하는지 확인할 수 있습니다.

```cedar
permit(principal, action, resource)
when {
    principal.hasTag("sub") &&
    principal.getTag("sub") == "your-client-id"
};
```

**참고**: Cognito M2M 액세스 토큰의 `sub` 클레임은 앱 클라이언트의 `client_id`와 같습니다. `getTag("sub")`를 확인하면 호출하는 principal을 고유하게 식별할 수 있으며, 이는 principal ID 확인과 동일합니다.

In [ ]:
# 이전 정책 정리
print("=" * 70)
print("TEST SCENARIO 3: Principal ID-Based Access Control")
print("=" * 70)

print("\nCleaning up previous policies...")
for pid in CREATED_POLICIES:
    delete_policy(pid)
CREATED_POLICIES.clear()

### Step 7.1: Client ID 검증용 Cedar 정책 생성

In [ ]:
policy_name = f"principal_id_policy_{int(time.time())}"

# Client Credentials Flow에서 'sub' 클레임은 client_id와 같으며
# getTag("sub")를 통해 principal 태그로 액세스할 수 있음
cedar_statement = f'''permit(principal,
    action == AgentCore::Action::"RefundToolTarget___refund",
    resource == AgentCore::Gateway::"{GATEWAY_ARN}")
when {{
    principal.hasTag("sub") &&
    principal.getTag("sub") == "{CLIENT_ID}"
}};'''

print(f"Cedar statement:\n{cedar_statement}")

policy_id = create_cedar_policy(
    policy_name=policy_name,
    cedar_statement=cedar_statement,
    description=f"Allow requests only from principal with sub (client_id): {CLIENT_ID}",
)

if policy_id:
    CREATED_POLICIES.append(policy_id)

    # 정책이 ACTIVE 상태가 될 때까지 대기
    print("\n⏳ Waiting for policy to become ACTIVE...")
    if wait_for_policy_active(policy_id):
        print("✓ Policy is ACTIVE and ready for testing")
    else:
        print("\n⚠️  Policy did not become ACTIVE. Tests may fail.")

### Step 7.2: 일치하는 Principal ID로 테스트(예상 결과: ALLOWED)

In [ ]:
print("\n" + "=" * 70)
print(f"Test 3.1: Request with sub (client_id)='{CLIENT_ID}'")
print("=" * 70)

token = get_bearer_token()
claims = decode_token(token)
# Client Credentials Flow에서 'sub' 클레임은 client_id와 같으며 principal 태그 "sub"에 매핑됨
print(f"\nToken 'sub' claim (used as getTag('sub')): {claims.get('sub')}")
print(f"Token 'client_id' claim: {claims.get('client_id')}")

result = make_gateway_request(
    bearer_token=token,
    tool_name="RefundToolTarget___refund",
    arguments={"amount": 500, "orderId": "test-principal-id"},
)

print("\nRequest: RefundToolTarget___refund(amount=500)")
print("\nResponse:")
print(json.dumps(result, indent=2))

outcome = analyze_response(result)
display_test_result("ALLOWED", outcome, "Matching principal sub (client_id) should be ALLOWED")

print("\n💡 Note: To test DENY scenario, you would need a different")
print("   Amazon Cognito app client with a different client_id (sub)")

---

## Part 8: 고급 패턴

### 여러 조건 결합

더 복잡한 액세스 제어 시나리오에서는 하나의 정책에 여러 조건을 결합할 수 있습니다.

In [ ]:
# 예: 결합 정책(부서 AND 금액 한도)
print("\nAdvanced Pattern: Combined Conditions")
print("=" * 70)

combined_cedar = f'''permit(principal,
    action == AgentCore::Action::"RefundToolTarget___refund",
    resource == AgentCore::Gateway::"{GATEWAY_ARN}")
when {{
    principal.hasTag("department_name") &&
    principal.getTag("department_name") == "finance" &&
    context.input.amount <= 1000
}};'''

print("Cedar Policy with Combined Conditions:")
print("-" * 60)
print(combined_cedar)
print("-" * 60)
print("\nThis policy allows requests when:")
print("  ✓ User is in finance department")
print("  ✓ AND refund amount is <= $1000")

### `like` 연산자를 사용한 패턴 일치

`like` 연산자는 유연한 일치를 위해 와일드카드를 지원합니다.

| 패턴 | 일치 조건 |
|---------|--------|
| `"*admin*"` | 어느 위치든 "admin"을 포함 |
| `"admin*"` | "admin"으로 시작 |
| `"*admin"` | "admin"으로 끝남 |
| `"team-*"` | "team-"으로 시작 |

In [ ]:
# 예: 팀 기반 액세스를 위한 패턴 일치
print("\nAdvanced Pattern: Team-Based Access with Wildcards")
print("=" * 70)

team_cedar = f'''permit(principal,
    action == AgentCore::Action::"RefundToolTarget___refund",
    resource == AgentCore::Gateway::"{GATEWAY_ARN}")
when {{
    principal.hasTag("groups") &&
    principal.getTag("groups") like "*team-finance*"
}};'''

print("Cedar Policy with Pattern Matching:")
print("-" * 60)
print(team_cedar)
print("-" * 60)
print("\nThis policy allows requests when:")
print("  ✓ User's groups contain 'team-finance'")
print("  ✓ Matches: ['team-finance', 'developers']")
print("  ✓ Matches: ['admins', 'team-finance-leads']")

---

## Part 9: 모범 사례

### 정책 설계 모범 사례

1. **구체적인 action 사용** - 와일드카드 대신 특정 도구 action을 대상으로 지정합니다.
2. **항상 클레임 존재 여부 확인** - 오류를 방지하려면 `getTag()` 전에 `hasTag()`를 사용합니다.
3. **패턴 일치 신중하게 사용** - `like "*value*"`는 의도하지 않은 문자열과 일치할 수 있습니다.
4. **ALLOW와 DENY 모두 테스트** - 정책이 양방향으로 올바르게 작동하는지 확인합니다.
5. **정책 문서화** - 의미를 알 수 있는 이름과 설명을 사용합니다.

### Amazon Cognito 구성 모범 사례

1. **V3_0 트리거 사용** - M2M 클라이언트 자격 증명 흐름에 필요합니다.
2. **Essentials 티어로 업그레이드** - V3_0에는 Amazon Cognito Essentials 또는 Plus가 필요합니다.
3. **토큰 클레임 테스트** - 정책을 생성하기 전에 항상 토큰에 클레임이 표시되는지 확인합니다.
4. **배열을 신중하게 처리** - JWT 클레임에서 배열은 문자열로 직렬화됩니다.

### 피해야 할 일반적인 실수

- ❌ M2M 흐름에서 V1_0 또는 V2_0 트리거 사용(클레임이 추가되지 않음)
- ❌ `getTag()` 전에 `hasTag()` 확인 누락
- ❌ 배열에 패턴 일치(`like`)가 필요한 경우 정확히 일치(`==`) 사용
- ❌ 테스트 전에 정책이 ACTIVE 상태가 될 때까지 기다리지 않음
- ❌ 적절한 정리 전략 없이 정책 생성

### 필수 AWS IAM 권한

#### Policy 관리용

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": [
        "policy-registry:CreatePolicyEngine",
        "policy-registry:GetPolicyEngine",
        "policy-registry:ListPolicyEngines",
        "policy-registry:CreatePolicy",
        "policy-registry:DeletePolicy",
        "policy-registry:ListPolicies"
      ],
      "Resource": "*"
    }
  ]
}
```

#### Amazon Cognito AWS Lambda 트리거용

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": [
        "cognito-idp:UpdateUserPool",
        "lambda:CreateFunction",
        "lambda:UpdateFunctionCode",
        "lambda:AddPermission",
        "iam:CreateRole",
        "iam:AttachRolePolicy",
        "iam:PassRole"
      ],
      "Resource": "*"
    }
  ]
}
```

---

## Part 10: 리소스 정리

이 튜토리얼에서 생성한 모든 정책을 삭제합니다.

In [ ]:
print("=" * 70)
print("CLEANUP")
print("=" * 70)

print(f"\nDeleting {len(CREATED_POLICIES)} policies...")
for pid in CREATED_POLICIES:
    delete_policy(pid)

CREATED_POLICIES.clear()
print("\n✓ Cleanup complete")

### 선택 사항: 모든 정책 삭제

Policy Engine의 모든 정책을 정리할 때 사용합니다. 주의해서 사용하세요.

In [ ]:
# Policy Engine의 모든 정책을 삭제하려면 주석 해제
# 경고: 이 튜토리얼에서 생성한 정책뿐만 아니라 모든 정책이 삭제됨

# print("Deleting ALL policies...")
# policies = list_policies()
# for policy in policies:
#     policy_id = policy.get('policyId')
#     if policy_id:
#         delete_policy(policy_id)
# print("✓ All policies deleted")

---

## 마무리

축하합니다! Policy for Amazon Bedrock AgentCore 튜토리얼을 완료했습니다. 다음 내용을 학습했습니다.

✅ JWT 토큰에 사용자 지정 클레임을 추가하도록 Amazon Cognito AWS Lambda 트리거 구성  
✅ principal 태그를 통해 JWT 클레임을 검증하는 Cedar 정책 생성  
✅ 부서 기반 액세스 제어 구현  
✅ 패턴 일치를 사용하는 그룹 기반 액세스 제어 구현  
✅ Principal ID 기반 액세스 제어 구현  
✅ 복잡한 액세스 제어 시나리오를 위한 여러 조건 결합  

### 주요 Cedar 구문 패턴

| 클레임 유형 | Cedar 구문 |
|------------|-------------|
| 문자열(정확히 일치) | `principal.getTag("claim") == "value"` |
| 문자열(포함) | `principal.getTag("claim") like "*value*"` |
| 배열(포함) | `principal.getTag("claim") like "*value*"` |
| 입력값 검증 | `context.input.field <= value` |

### 다음 단계

1. **프로덕션에 구현** - Amazon Bedrock AgentCore 배포에 이러한 패턴을 적용합니다.
2. **정책 사용자 지정** - 구체적인 액세스 제어 요구 사항에 맞게 정책을 조정합니다.
3. **모니터링 추가** - 정책 거부에 대한 Amazon CloudWatch 경보를 설정합니다.
4. **반복 개선** - 실제 사용 사례를 기반으로 정책을 구체화합니다.

---

**작성자**: AWS  
**라이선스**: MIT-0  
**마지막 업데이트**: 2025